In [186]:
import torch

print("PyTorch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

print("Using:", device)

PyTorch: 2.14.1
MPS available: True
Using: mps


## 初始化与MiniMax

In [187]:
import numpy as np


def initial_state():
    board = np.zeros((3, 3), dtype=int)  # 提示：np.zeros((行数, 列数), dtype=int)
    current_player = 1
    return board, current_player


# board, player = initial_state()

# board[1, 1] = 1   # 把当前玩家的棋子放到中央
# player = -1            # 切换到另一位玩家

# for row, col in [(1, 1), (0, 0)]:
#     board[row, col] = player   # 哪个变量存着这一手的棋子值？
#     player = -player            # 怎样把当前值的正负号翻转？

# for row, col in [(1, 1), (1, 1)]:
#     if board[row, col] == 0:
#         board[row, col] = player
#         player = -player

In [188]:
def legal_actions(board):
    moves = list()
    for row in range(3):
        for col in range(3):
            if board[row, col] == 0:
                moves.append((row, col))
    return moves

In [189]:
def has_row_win(board, player):
    for row in range(3):
        if np.all(board[row, :] == player):
            return True
    return False

In [190]:
def has_col_win(board, player):
    for col in range(3):
        if np.all(board[:, col] == player):
            return True
    return False

In [191]:
# def has_diag_win(board, player):
#     for i in np.diag(board):
#         if i != player:
#             for i in np.diag(np.fliplr(board)):
#                 if i != player:
#                     return False
#     return True
def has_diag_win(board, player):
    main_win = np.all(np.diag(board) == player)
    anti_win = np.all(np.diag(np.fliplr(board)) == player)
    return main_win or anti_win

In [192]:
def has_won(board, player):
    row = has_row_win(board, player)
    col = has_col_win(board, player)
    diag = has_diag_win(board, player)
    return row or col or diag

In [193]:
def game_result(board, player):
    if has_won(board, -player) == True:
        return -player
    elif len(legal_actions(board)) == 0:
        return 0
    return None

In [194]:
def step(board, player, row, col):
    result = game_result(board, player)
    if result != None:
        return board, player
    if 0 <= row < 3 and 0 <= col < 3 and board[row, col] == 0:
        board[row, col] = player
        player = -player
    return board, player

In [195]:
import random


def random_action(board):
    moves = random.choice(legal_actions(board))
    return moves

In [196]:
def play_random_game():
    board, player = initial_state()

    while game_result(board, player) is None:
        row, col = random_action(board)
        board, player = step(board, player, row, col)

    print(board)
    return game_result(board, player)

In [197]:
# 终局评分
def terminal_value(result, player):
    if result == 0:
        return 0
    elif result == player:
        return 1
    else:
        return -1

In [198]:
# 从当前局面出发，假设双方都走得最好，轮到行动的玩家最终能拿几分？当棋盘上有好几个空格时，这个函数就更有用了。它会试走每个合法位置，再考虑对手会怎样应对。每条走法一旦走到终局，你写的 terminal_value() 就能给它打分。
def minimax_value(board, player):
    result = game_result(board, player)
    if result is not None:
        return terminal_value(result, player)
    scores = []
    for row, col in legal_actions(board):  # 是否有问题，遍历怎么能称为“走得最好”
        child_board, next_player = step(board.copy(), player, row, col)
        child_score = minimax_value(child_board, next_player)
        scores.append(-child_score)
    return max(scores)

In [199]:
def minimax_action(board, player):
    moves = legal_actions(board)
    scores = []
    for row, col in moves:
        child_board, next_player = step(board.copy(), player, row, col)
        child_score = minimax_value(child_board, next_player)
        scores.append(-child_score)

    best_index = scores.index(max(scores))
    return moves[best_index]

In [200]:
def play_minimax_vs_random():
    board, player = initial_state()

    while game_result(board, player) is None:
        if player == -1:
            row, col = minimax_action(board, player)  # O 用 Minimax
        else:
            row, col = random_action(board)  # X 随机选
        board, player = step(board, player, row, col)

    print(board)
    return game_result(board, player)

In [201]:
def play_minimax_vs_minimax():
    board, player = initial_state()

    while game_result(board, player) is None:
        row, col = minimax_action(board, player)
        board, player = step(board, player, row, col)

    print(board)
    return game_result(board, player)

## MTCS

In [202]:
# 从给定棋盘下，对输入玩家来说，随机下能有什么分数
# 复制棋盘，让双方随机下到终局，返回起始玩家视角的得分，赢 +1、平 0、输 -1。
def rollout(board, player):
    board = board.copy()
    start_player = player

    while game_result(board, player) is None:
        row, col = random_action(board)
        board, player = step(board, player, row, col)

    result = game_result(board, player)

    return terminal_value(result, start_player)

In [203]:
board99 = np.array(
    [
        [1, 0, 1],  # X . X
        [1, 1, -1],  # X X O
        [-1, -1, 0],  # O O .
    ]
)
N = 0
W = 0
for i in range(20):
    N = N + 1
    W = W + rollout(board99, -1)
Q = W / N  # 得分除局数，平均得分
print(N, W, Q)

20 -2 -0.1


In [204]:
# 对moves单独打印得分
moves99 = legal_actions(board99)
for row, col in moves99:
    child_board, next_player = step(board99.copy(), -1, row, col)
    N = 0
    W = 0
    for i in range(5):
        N = N + 1
        W = W - rollout(child_board, next_player)
    Q = W / N
    print((row, col), N, W, Q)

(0, 1) 5 -5 -1.0
(2, 2) 5 5 1.0


In [205]:
child_board, next_player = step(board99.copy(), -1, 0, 1)
child_score = rollout(child_board, next_player)

score = -child_score

print(child_score, score)

1 -1


In [206]:
stats = {
    (0, 1): {"N": 0, "W": 0},
    (2, 2): {"N": 0, "W": 0},
}

record = stats[(0, 1)]

record["N"] = record["N"] + 1
record["W"] = record["W"] + score
print(stats[(0, 1)])

{'N': 1, 'W': -1}


In [207]:
# 	先走指定动作，再调用 rollout。把子局面得分取负，转换为落子方的得分。
def simulate_move(board, player, move, stats):
    row, col = move
    child_board, next_player = step(board.copy(), player, row, col)

    child_score = rollout(child_board, next_player)
    score = -child_score

    record = stats[move]
    record["N"] = record["N"] + 1
    record["W"] = record["W"] + score

In [208]:
stats = {
    (0, 1): {"N": 0, "W": 0},
    (2, 2): {"N": 0, "W": 0},
}

simulate_move(board99, -1, (2, 2), stats)
print(stats[(2, 2)])

{'N': 1, 'W': 1}


In [209]:
stats = {
    (0, 1): {"N": 0, "W": 0},
    (2, 2): {"N": 0, "W": 0},
}

simulate_move(board99, -1, (2, 2), stats)
print(stats[(2, 2)])

simulate_move(board99, -1, (2, 2), stats)
print(stats[(2, 2)])

{'N': 1, 'W': 1}
{'N': 2, 'W': 2}


In [210]:
simulate_move(board99, -1, (0, 1), stats)

for move in legal_actions(board99):
    record = stats[move]
    Q = record["W"] / record["N"]
    print(move, Q)

(0, 1) -1.0
(2, 2) 1.0


In [211]:
moves = legal_actions(board99)
scores = []

for move in moves:
    record = stats[move]
    Q = record["W"] / record["N"]
    scores.append(Q)

best_index = scores.index(max(scores))

print(moves[best_index])

(2, 2)


In [212]:
stats = {
    (0, 1): {"N": 1, "W": -1},
    (2, 2): {"N": 0, "W": 0},
}

for move in legal_actions(board99):
    record = stats[move]
    if record["N"] == 0:
        simulate_move(board99, -1, move, stats)

In [213]:
print(stats[(0, 1)])
print(stats[(2, 2)])

{'N': 1, 'W': -1}
{'N': 1, 'W': 1}


In [214]:
Q = 0.5
total_N = 22
N = 2

bonus = np.sqrt(np.log(total_N) / N)

uct_score = Q + bonus

print(round(uct_score, 3))

1.743


In [215]:
total_N = 0

for move in legal_actions(board99):
    record = stats[move]
    total_N = total_N + record["N"]

print(total_N)

2


In [216]:
for move in legal_actions(board99):
    record = stats[move]
    N = record["N"]
    Q = record["W"] / N

    bonus = np.sqrt(np.log(total_N) / N)
    uct_score = Q + bonus

    print(move, round(uct_score, 3))

(0, 1) -0.167
(2, 2) 1.833


In [217]:
moves = legal_actions(board99)
scores = []

for move in moves:
    record = stats[move]
    N = record["N"]
    Q = record["W"] / N
    bonus = np.sqrt(np.log(total_N) / N)
    uct_score = Q + bonus

    scores.append(uct_score)


best_index = scores.index(max(scores))
next_move = moves[best_index]
print(next_move)

(2, 2)


In [218]:
simulate_move(board99, -1, next_move, stats)
# 补一行，调用 simulate_move


print(next_move, stats[next_move])

(2, 2) {'N': 2, 'W': 2}


In [219]:
def select_uct(board, stats):
    moves = legal_actions(board)

    for move in moves:
        record = stats[move]
        if record["N"] == 0:
            return move

In [220]:
test_stats = {
    (0, 1): {"N": 0, "W": 0},
    (2, 2): {"N": 0, "W": 0},
}

print(select_uct(board99, test_stats))

(0, 1)


In [221]:
test_stats = {
    (0, 1): {"N": 0, "W": 0},
    (2, 2): {"N": 0, "W": 0},
}

board999 = np.array(
    [
        [1, 0, 1],  # X . X
        [1, 1, -1],  # X X O
        [-1, -1, 0],  # O O .
    ]
)

move5 = select_uct(board999, test_stats)

simulate_move(board999, -1, move5, test_stats)

print(move5, test_stats[move5])

(0, 1) {'N': 1, 'W': -1}


In [222]:
# 优先选择未模拟过的动作，其余情况根据 Q + bonus 安排下一次模拟。
def select_uct(board, stats):
    moves = legal_actions(board)
    total_N = 0

    # 优先选择未模拟过的动作
    for move in moves:
        record = stats[move]
        if record["N"] == 0:
            return move
        total_N = total_N + record["N"]

    # 所有动作都已模拟过，比较 UCT 评分
    scores = []
    for move in moves:
        record = stats[move]
        N = record["N"]
        Q = record["W"] / N
        bonus = np.sqrt(np.log(total_N) / N)
        uct_score = Q + bonus
        scores.append(uct_score)

    best_index = scores.index(max(scores))
    return moves[best_index]

In [223]:
visited_stats = {
    (0, 1): {"N": 1, "W": -1},
    (2, 2): {"N": 1, "W": 1},
}

print(select_uct(board999, visited_stats))

(2, 2)


In [224]:
search_stats = {
    (0, 1): {"N": 0, "W": 0},
    (2, 2): {"N": 0, "W": 0},
}

for i in range(3):
    move = select_uct(board999, search_stats)
    simulate_move(board999, -1, move, search_stats)
    print(i + 1, move, search_stats[move])

1 (0, 1) {'N': 1, 'W': -1}
2 (2, 2) {'N': 1, 'W': 1}
3 (2, 2) {'N': 2, 'W': 2}


In [225]:
moves = legal_actions(board999)
visits = []

for move in moves:
    record = search_stats[move]
    visits.append(record["N"])
    # 补一行，把这个动作的 N 追加到 visits

best_index = visits.index(max(visits))
best_move = moves[best_index]

print(visits)
print(best_move)

[1, 2]
(2, 2)


In [226]:
# 初始化统计，重复选择与模拟，最后返回访问次数最多的动作。
def search_action(board, player, num_simulations):
    if game_result(board, player) is not None:
        return None

    # 为当前棋盘的每个合法动作建立记录
    moves = legal_actions(board)
    stats = {}
    for move in moves:
        stats[move] = {"N": 0, "W": 0}

    # 反复选择动作、模拟并更新记录
    for i in range(num_simulations):
        move = select_uct(board, stats)
        # 调用 select_uct，把动作保存到 move
        simulate_move(board, player, move, stats)
        # 调用 simulate_move，更新 stats

    # 根据最终访问次数推荐动作
    visits = []
    for move in moves:
        record = stats[move]
        visits.append(record["N"])

    best_index = visits.index(max(visits))
    return moves[best_index]

In [227]:
before = board999.copy()

print(search_action(board999, -1, 3))  # 预期 (2, 2)
print(np.array_equal(board999, before))  # 预期 True

(2, 2)
True


In [228]:
before = board999.copy()

move = search_action(board999, -1, 3)
row, col = move
after_board, next_player = step(board999.copy(), -1, row, col)

print(move)
print(game_result(after_board, next_player))
print(np.array_equal(board999, before))

(2, 2)
-1
True


In [229]:
root = {
    "board": board999.copy(),
    "player": -1,
    "children": {},
}

move = (0, 1)
row, col = move
child_board, next_player = step(root["board"].copy(), root["player"], row, col)

child = {
    "board": child_board,
    "player": next_player,
    "children": {},
}

root["children"][move] = child

print(root["children"][move]["board"])
print(root["children"][move]["player"])
print(root)

[[ 1 -1  1]
 [ 1  1 -1]
 [-1 -1  0]]
1
{'board': array([[ 1,  0,  1],
       [ 1,  1, -1],
       [-1, -1,  0]]), 'player': -1, 'children': {(0, 1): {'board': array([[ 1, -1,  1],
       [ 1,  1, -1],
       [-1, -1,  0]]), 'player': 1, 'children': {}}}}


In [230]:
move = (2, 2)
row, col = move
child_board, next_player = step(root["board"].copy(), root["player"], row, col)

child = {
    "board": child_board,
    "player": next_player,
    "children": {},
}

root["children"][move] = child

print(len(root["children"]))
print(list(root["children"]))

2
[(0, 1), (2, 2)]


In [231]:
def make_node(board, player):
    stats = {}

    if game_result(board, player) is None:
        for move in legal_actions(board):
            stats[move] = {"N": 0, "W": 0}  # 补一行，为 move 创建初始统计

    return {
        "board": board.copy(),
        "player": player,
        "stats": stats,
        "children": {},
    }

In [232]:
saved_child = root["children"][(0, 1)]
test_node = make_node(saved_child["board"], saved_child["player"])

print(test_node["player"])
print(test_node["stats"])

1
{(2, 2): {'N': 0, 'W': 0}}


In [233]:
def expand(node, move):
    if move in node["children"]:
        return node["children"][move]

    row, col = move
    child_board, next_player = step(node["board"].copy(), node["player"], row, col)

    child = make_node(
        child_board, next_player
    )  # 补一行，用 child_board 和 next_player 创建 child

    node["children"][move] = child
    return child

In [234]:
root = make_node(board999, -1)
child = expand(root, (0, 1))

print(child["player"])
print(child["stats"])
print(child["children"])
print(expand(root, (0, 1)) is child)

1
{(2, 2): {'N': 0, 'W': 0}}
{}
True


In [235]:
root = make_node(board999, -1)

root_move = select_uct(root["board"], root["stats"])
child = expand(root, root_move)

child_move = select_uct(child["board"], child["stats"])
# 调用 select_uct，使用 child 的棋盘和统计，得到 child_move
leaf = expand(child, child_move)
# 调用 expand，展开 child_move，得到 leaf

result = game_result(leaf["board"], leaf["player"])

print(root_move, child_move)
print(result)
print(leaf["stats"])

(0, 1) (2, 2)
1
{}


In [236]:
root = make_node(board999, -1)
root_move = select_uct(root["board"], root["stats"])
child = expand(root, root_move)
child_move = select_uct(child["board"], child["stats"])
leaf = expand(child, child_move)

result = game_result(leaf["board"], leaf["player"])
leaf_score = terminal_value(result, leaf["player"])

child_score = -leaf_score
# 补一行，把 leaf_score 换成 child 视角的 child_score
root_score = -child_score
# 补一行，把 child_score 换成 root 视角的 root_score

child_record = child["stats"][child_move]
child_record["N"] = child_record["N"] + 1
child_record["W"] = child_record["W"] + child_score

root_record = root["stats"][root_move]
root_record["N"] = root_record["N"] + 1
root_record["W"] = root_record["W"] + root_score

print("child", child_record)
print("root", root_record)

child {'N': 1, 'W': 1}
root {'N': 1, 'W': -1}


In [237]:
def backpropagate(path, leaf_score):
    score = leaf_score

    for node, move in reversed(path):
        score = -score
        # 补一行，将 score 换成当前 node 的玩家视角

        record = node["stats"][move]
        record["N"] = record["N"] + 1
        record["W"] = record["W"] + score

In [238]:
root = make_node(board999, -1)
child = expand(root, (0, 1))
leaf = expand(child, (2, 2))

path = [
    (root, (0, 1)),  # O 在根局面选择的动作
    (child, (2, 2)),  # X 在子局面选择的动作
]

leaf_score = rollout(leaf["board"], leaf["player"])
backpropagate(path, leaf_score)

print("child", child["stats"][(2, 2)])
print("root", root["stats"][(0, 1)])

child {'N': 1, 'W': 1}
root {'N': 1, 'W': -1}


In [239]:
def mcts_simulation(root):
    node = root
    path = []

    while game_result(node["board"], node["player"]) is None:
        move = select_uct(node["board"], node["stats"])
        path.append((node, move))

        if move in node["children"]:
            node = node["children"][move]
        else:
            node = expand(node, move)
            break

    leaf_score = rollout(node["board"], node["player"])
    # 调用 rollout，使用当前 node 的棋盘和玩家，得到 leaf_score
    backpropagate(path, leaf_score)
    # 调用 backpropagate，使用 path 和 leaf_score 更新统计

In [240]:
root = make_node(board999, -1)
child = expand(root, (0, 1))

mcts_simulation(root)

print("child", child["stats"][(2, 2)])
print("root", root["stats"][(0, 1)])

child {'N': 1, 'W': 1}
root {'N': 1, 'W': -1}


In [241]:
def search_action(board, player, num_simulations):
    if game_result(board, player) is not None:
        return None

    root = make_node(board, player)

    for i in range(num_simulations):
        mcts_simulation(root)
        # 补一行，以 root 为起点运行一次 mcts_simulation

    moves = legal_actions(board)
    visits = []

    for move in moves:
        record = root["stats"][move]
        visits.append(record["N"])

    best_index = visits.index(max(visits))
    return moves[best_index]

In [242]:
before = board999.copy()

print(search_action(board999, -1, 3))
print(np.array_equal(board999, before))

(2, 2)
True


In [243]:
import random

defense_board = np.array(
    [
        [1, 1, 0],  # X X .
        [0, -1, 0],  # . O .
        [0, 0, 0],  # . . .
    ]
)
before = defense_board.copy()

print("Minimax", minimax_action(defense_board, -1))

random.seed(0)
print("MCTS", search_action(defense_board, -1, 500))
print(np.array_equal(defense_board, before))

Minimax (0, 2)
MCTS (0, 2)
True


In [244]:
reference_move = minimax_action(defense_board, -1)

for num_simulations in [10, 50, 100, 500]:
    correct_count = 0

    for seed in range(10):
        random.seed(seed)

        move = search_action(defense_board, -1, num_simulations)
        # 调用 search_action，得到 move

        if move == reference_move:
            correct_count = correct_count + 1

    print(num_simulations, correct_count, "/10")

10 8 /10
50 10 /10
100 10 /10
500 10 /10


In [245]:
def play_mcts_vs_random(num_simulations):
    board, player = initial_state()

    while game_result(board, player) is None:
        if player == 1:
            row, col = search_action(board, player, num_simulations)
            # 调用 search_action，得到 row、col
        else:
            row, col = random_action(board)

        board, player = step(board, player, row, col)

    return board, game_result(board, player)

In [246]:
random.seed(0)

final_board, result = play_mcts_vs_random(100)
print(final_board)
print(result)

[[-1  0  1]
 [ 0 -1 -1]
 [ 1  1  1]]
1


In [247]:
counts = {1: 0, 0: 0, -1: 0}

for seed in range(10):
    random.seed(seed)
    final_board, result = play_mcts_vs_random(100)

    counts[result] = counts[result] + 1
    # 补一行，累计本局赛果

print(counts)

{1: 10, 0: 0, -1: 0}


In [248]:
def play_mcts_vs_random(num_simulations, mcts_player=1):
    board, player = initial_state()

    while game_result(board, player) is None:
        if player == mcts_player:  # 修改这一行，使用 mcts_player 判断
            row, col = search_action(board, player, num_simulations)
        else:
            row, col = random_action(board)

        board, player = step(board, player, row, col)

    return board, game_result(board, player)

In [249]:
random.seed(0)

final_board, result = play_mcts_vs_random(100, mcts_player=-1)
print(final_board)
print(result)

[[-1  0  1]
 [ 1 -1  0]
 [ 1  0 -1]]
-1


In [250]:
for mcts_player in [1, -1]:
    # 每换一次执棋方，重新统计
    counts = {1: 0, 0: 0, -1: 0}

    for seed in range(10):
        random.seed(seed)

        # 完整下一局，得到原始赛果
        final_board, result = play_mcts_vs_random(100, mcts_player)

        # 换成 MCTS 的视角：赢 1，平 0，输 -1
        score = terminal_value(result, mcts_player)

        counts[score] = counts[score] + 1

    print("MCTS执棋方", mcts_player, counts)

MCTS执棋方 1 {1: 10, 0: 0, -1: 0}
MCTS执棋方 -1 {1: 10, 0: 0, -1: 0}


In [251]:
def play_mcts_vs_minimax(num_simulations, mcts_player=1):
    board, player = initial_state()

    while game_result(board, player) is None:
        if player == mcts_player:
            row, col = search_action(board, player, num_simulations)
        else:
            row, col = minimax_action(board, player)

        board, player = step(board, player, row, col)

    return board, game_result(board, player)

In [252]:
random.seed(0)

final_board, result = play_mcts_vs_minimax(100, mcts_player=1)

print(final_board)
print(result)

[[-1 -1  1]
 [ 1  1 -1]
 [-1  1  1]]
0


In [253]:
# for mcts_player in [1, -1]:
#     counts = {1: 0, 0: 0, -1: 0}

#     for seed in range(10):
#         random.seed(seed)

#         final_board, result = play_mcts_vs_minimax(500, mcts_player)
#         score = terminal_value(result, mcts_player)

#         counts[score] = counts[score] + 1

#     print("MCTS执棋方", mcts_player, counts)

## 双头网络

In [254]:
import numpy as np


# 将棋盘按照player拆成两个，输出一个+1维的numpy数组
def encode_board(board, player):
    own_plane = (board == player).astype(np.float32)
    opponent_plane = (board == -player).astype(np.float32)
    return np.stack([own_plane, opponent_plane], axis=0)


defense_board = np.array(
    [
        [1, 1, 0],
        [0, -1, 0],
        [0, 0, 0],
    ]
)

encoded = encode_board(defense_board, player=-1)
print(encoded.shape, encoded.dtype, type(encoded))
# print(encoded[0])
# print(encoded[1])
# print(encoded)

(2, 3, 3) float32 <class 'numpy.ndarray'>


In [255]:
import torch


# 讲numpy数组变成tensor，且再+1维。unsqueeze只能加1维，即套一层壳。player是谁就把谁的视角的board放在前面
def board_to_input(board, player):
    encoded = encode_board(board, player)  # NumPy 数组，形状 (2, 3, 3)
    x = torch.from_numpy(encoded)  # 转成 PyTorch 张量，形状和数值都保留
    return x.unsqueeze(0)  # 在指定位置插入一个长度为 1 的维度


x = board_to_input(defense_board, player=-1)
# print(x.shape, x.dtype)
# print(x[0, :, 0, 0].tolist())
print(x)

tensor([[[[0., 0., 0.],
          [0., 1., 0.],
          [0., 0., 0.]],

         [[1., 1., 0.],
          [0., 0., 0.],
          [0., 0., 0.]]]])


In [ ]:
from torch import nn

# 把每个局面的 2 × 3 × 3 个数展平成 18 个数，再用一层 nn.Linear(18, 32) 把它们组合成 32 个特征。这里的 32 是我们给小网络选的宽度。随后，两个头各自读取这组特征，回答自己的问题。让输入沿这些层算出输出，就叫一次前向传播。
shared_layer = nn.Linear(18, 32)
policy_head = nn.Linear(
    32, 9
)  # 策略头给每个格子一个分数，供搜索判断哪些走法值得优先探索。
value_head = nn.Linear(
    32, 1
)  # 价值头给整个局面一个分数，估计从当前行动方 O 的角度，走到终局能拿多少分。为什么是O，因为例子用的defence_board马上要O走了。

flat = x.flatten(start_dim=1)  # 保留第 0 维的局面数，（1，2，3，3），从“2”开始展开
features = torch.relu(shared_layer(flat))  # ReLU 把负数置零，保留正数(激活函数)
logits = policy_head(features)
value = torch.tanh(value_head(features))  # 把估值限制在 [-1, 1](激活函数)

print(logits.shape, value.shape)
print("右上角的落子分数", logits[0, 2].item())
print("从 O 看整盘棋的估值", value[0, 0].item())
print(logits)
print(value_head(features))

torch.Size([1, 9]) torch.Size([1, 1])
右上角的落子分数 0.11942798644304276
从 O 看整盘棋的估值 -0.10254637151956558
tensor([[-0.1322,  0.1042,  0.1194, -0.1748,  0.1824,  0.0970, -0.1645, -0.0477,
          0.2493]], grad_fn=<AddmmBackward0>)
tensor([[-0.1029]], grad_fn=<AddmmBackward0>)


logits相较于value少做了一次激活，这是故意设计的吗
后续要做softmax，激活后会破坏；value_head要把估值限制在 [-1, 1]。

In [ ]:
# 把logits这九个分数变成九个概率：已占格的概率为 0，所有空格的概率加起来是 1。
def legal_policy(board, move_logits):
    cells = torch.as_tensor(
        board, device=move_logits.device
    ).flatten()  # torch.as_tensor(...)：PyTorch 工厂函数，将 board（可能是 list、numpy array 或已有 tensor）转换为 PyTorch Tensor。
    masked_logits = move_logits.masked_fill(
        cells != 0, float("-inf")
    )  # .masked_fill(condition, value)：PyTorch 方法，在 condition 为 True 的位置将值替换为 value，其余位置保持不变
    return torch.softmax(masked_logits, dim=0)


test_logits = torch.zeros_like(
    logits[0]
)  # 一个局面的九个分数。torch.zeros_like(tensor)：PyTorch 工厂函数，创建一个与输入张量形状、dtype、device 完全相同的全零张量
test_logits[0] = 10.0  # 左上角，已有 X（测试用）
test_logits[2] = 1.0  # 右上角，空格（测试用）

priors = legal_policy(defense_board, test_logits)
print(priors)
print(priors.reshape(3, 3))
print(priors.sum().item())

tensor([0.0000, 0.0000, 0.3522, 0.1296, 0.0000, 0.1296, 0.1296, 0.1296, 0.1296])
tensor([[0.0000, 0.0000, 0.3522],
        [0.1296, 0.0000, 0.1296],
        [0.1296, 0.1296, 0.1296]])
1.0


In [258]:
# 前向计算
def forward_batch(x_batch):
    flat = x_batch.flatten(start_dim=1)
    features = torch.relu(shared_layer(flat))
    logits = policy_head(features)
    values = torch.tanh(value_head(features))
    return logits, values


# step 会原地落子，传入副本才能同时保留落子前后的棋盘
after_board, after_player = step(defense_board.copy(), -1, 0, 2)

before_x = board_to_input(defense_board, -1)
after_x = board_to_input(after_board, after_player)
x_batch = torch.cat(
    [before_x, after_x], dim=0
)  # PyTorch 提供的张量拼接函数，用于将多个张量沿某一维度连接起来。

batch_logits, batch_values = forward_batch(
    x_batch
)  # 为什么多个batch可以进入shared_layer：nn.Linear(in_features, out_features) 接受形状为 (*, H_in) 的输入，其中 * 代表零个或多个任意大小的批量维度，H_in 必须严格等于 in_features。输出形状为 (*, H_out)。(2, 18) 的含义是表示一个包含 2 个样本的批次（batch），每个样本的特征维度为 18。这两个样本会在同一次前向传播中并行计算，共享同一组权重矩阵。
before_p = legal_policy(defense_board, batch_logits[0])
after_p = legal_policy(after_board, batch_logits[1])

print(x_batch.shape, batch_logits.shape, batch_values.shape)
print("右上角落子前后的概率", before_p[2].item(), after_p[2].item())
print(batch_values)

torch.Size([2, 2, 3, 3]) torch.Size([2, 9]) torch.Size([2, 1])
右上角落子前后的概率 0.18313370645046234 0.0
tensor([[-0.1025],
        [ 0.0934]], grad_fn=<TanhBackward0>)


In [ ]:
sample_board, turn = step(after_board.copy(), after_player, 2, 2)
sample_board, turn = step(sample_board, turn, 2, 0)
result = game_result(sample_board, turn)

players = [-1, after_player]  # 顺序与 x_batch 的两行一致
scores = [terminal_value(result, player) for player in players]
targets = batch_values.new_tensor(scores).reshape(
    -1, 1
)  # new_tensor(data) 是 PyTorch Tensor 对象的一个实例方法。以 batch_values 为模板，创建一个新张量。新张量会自动继承 batch_values 的以下属性：dtype（数据类型，如 float32）、device（设备，如 cuda:0 / cpu）、layout。reshape(shape) 也是 Tensor 的方法，返回一个指定形状的新视图（view）。-1 的含义：表示该维度的大小由 PyTorch 自动推断。

value_loss = (batch_values - targets).square().mean()

print("终局赛果", result)
print("两行的预测", batch_values.tolist())  # .tolist()任意形状的张量
print("两行的目标", targets.tolist())
print("价值损失", value_loss.item())  # .item()仅含1个元素的张量

终局赛果 -1
两行的预测 [[-0.10254637151956558], [0.09337453544139862]]
两行的目标 [[1.0], [-1.0]]
价值损失 1.2055381536483765


In [ ]:
shared_layer.zero_grad()  # 清掉累积的参数梯度，方便重跑这一格
value_head.zero_grad()

_, values = forward_batch(x_batch)
values.retain_grad()  # 把中间结果 values 的梯度也留下来，方便观察
value_loss = (values - targets).square().mean()

value_loss.backward()  # backward() 会从损失出发，沿着刚才的前向计算往回求导，一直算到价值头和共享层的每个参数。优化器下一步就能用这些梯度调整权重。

print("损失对 O、X 两个估值的梯度", values.grad.flatten().tolist())
# values.grad获取 values 张量的梯度（一个与 values 形状相同的 torch.Tensor）
print("价值头权重的梯度形状", value_head.weight.grad.shape)
# .weight：访问该模块的权重参数

损失对 O、X 两个估值的梯度 [-1.1025463342666626, 1.093374490737915]
价值头权重的梯度形状 torch.Size([1, 32])


In [ ]:
# 优化器怎么用不知道，背后什么原理也不知道
params = list(shared_layer.parameters()) + list(value_head.parameters())
optimizer = torch.optim.SGD(params, lr=0.01)

optimizer.zero_grad()  # 清掉上一格的梯度，这次用当前参数重新计算
_, values_before = forward_batch(x_batch)
loss_before = (values_before - targets).square().mean()
loss_before.backward()
optimizer.step()  # 用刚算出的参数梯度更新一次

with torch.no_grad():
    _, values_after = forward_batch(x_batch)
    loss_after = (values_after - targets).square().mean()

print("更新前", values_before.flatten().tolist(), loss_before.item())
print("更新后", values_after.flatten().tolist(), loss_after.item())

更新前 [-0.10254637151956558, 0.09337453544139862] 1.2055381536483765
更新后 [-0.0944024920463562, 0.07591081410646439] 1.1776504516601562


In [ ]:
# 面对这个局面，搜索更倾向于走哪一步
# 把每个格子分到了多少比例的访问也保留下来。
def policy_target_from_root(root):
    visits = torch.zeros((3, 3), dtype=torch.float32)  # 为什么要dtype=
    for move, record in root["stats"].items():
        visits[move] = record["N"]

    total = visits.sum()
    if total.item() == 0:
        raise ValueError("先运行至少一次模拟，再计算策略目标")
    return (visits / total).flatten()  # 展平成与策略头九个输出对应的顺序


two_empty = np.array(
    [
        [1, 0, 1],
        [1, 1, -1],
        [-1, -1, 0],
    ]
)
root = make_node(two_empty, -1)
for _ in range(3):
    mcts_simulation(root)

pi = policy_target_from_root(root)
print(pi)
print(pi.reshape(3, 3))
print(pi.sum().item())

tensor([0.0000, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.6667])
tensor([[0.0000, 0.3333, 0.0000],
        [0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.6667]])
1.0


In [ ]:
# w
def policy_loss_for_board(board, move_logits, target_pi):
    cells = torch.as_tensor(board, device=move_logits.device).flatten()
    legal = cells == 0
    log_p = torch.log_softmax(move_logits[legal], dim=0)
    pi_legal = target_pi.to(move_logits)[legal]
    return -(log_p * pi_legal).sum()


two_empty_logits, _ = forward_batch(board_to_input(two_empty, -1))
move_logits = two_empty_logits[0]
p = legal_policy(two_empty, move_logits)

equal_logits = torch.zeros_like(move_logits)
print(
    "两个合法动作等分时的损失",
    policy_loss_for_board(two_empty, equal_logits, pi).item(),
)
print("网络给上中、右下的概率", p[[1, 8]].tolist())
print("网络的策略损失", policy_loss_for_board(two_empty, move_logits, pi).item())

两个合法动作等分时的损失 0.6931471824645996
网络给上中、右下的概率 [0.44642090797424316, 0.5535790324211121]
网络的策略损失 0.6630648374557495
